<a href="https://colab.research.google.com/github/Decoding-Data-Science/sharjah/blob/main/notebooks/Day2_Simple_RAG_Gemini_LlamaIndex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple RAG with Gemini and LlamaIndex

**Workshop: Data and AI with Gemini | Day 2**

In this notebook you build a small RAG system on two airport PDF documents, look inside it, tune it, and put a chat window on top.

**RAG** means *Retrieval-Augmented Generation*: instead of asking the model from memory, we first **retrieve** the right pieces of our own documents, then let the model **generate** an answer from only those pieces.

## What you will do
1. Install the libraries and connect to Gemini (`gemini-3.5-flash-lite`) with your key
2. Load two PDF files from Google Drive
3. See what an **embedding** is
4. Build a vector index and **save it**, then open the saved files to see how it looks inside
5. Ask questions and look at the retrieved chunks
6. Test **chunking** (200, 300, 600) and **top_k** (1, 3, 5) with a small scoreboard
7. Lock in your settings and add a rule prompt
8. Put a **Gradio** chat app on top

Press **Shift + Enter** to run a cell. Run the cells in order.

## Step 1. Install the libraries

This takes about a minute the first time. The second cell prints the versions so you know it worked.

In [1]:
!pip install -q llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-readers-file pypdf gradio

In [2]:
import importlib.metadata as md
for package in ["llama-index-core", "llama-index-llms-google-genai", "llama-index-embeddings-google-genai", "gradio"]:
    print(package, md.version(package))

llama-index-core 0.14.25
llama-index-llms-google-genai 0.11.3
llama-index-embeddings-google-genai 0.6.0
gradio 6.31.0


## Step 2. Connect to Gemini

Your key comes from **Colab Secrets**, never from the notebook.
Click the **key icon** on the left, add a secret named `GEMINI_API_KEY`, paste your key, and switch on **Notebook access**.

We use two Gemini models:
- **`gemini-3.5-flash-lite`** writes the answers (the LLM).
- **`gemini-embedding-001`** turns text into numbers (the embedding model). We ask for **768 numbers** per text to keep things small.

In [ ]:
from google.colab import userdata
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from google.genai.types import EmbedContentConfig
import nest_asyncio

# Apply nest_asyncio to allow nested event loops in Jupyter notebooks
nest_asyncio.apply()

API_KEY = userdata.get("gemini3")

LLM_MODEL = "gemini-3.5-flash-lite"
EMBED_MODEL = "gemini-embedding-001"      # if your key cannot use it, try "gemini-embedding-2-preview"

Settings.llm = GoogleGenAI(model=LLM_MODEL, api_key=API_KEY)
Settings.embed_model = GoogleGenAIEmbedding(
    model_name=EMBED_MODEL,
    api_key=API_KEY,
    embedding_config=EmbedContentConfig(output_dimensionality=768),
)

# Quick test of both models
print(Settings.llm.complete("Say hello to the workshop in one short sentence."))
test_vector = Settings.embed_model.get_text_embedding("departure immigration queue")
print("The embedding model returned", len(test_vector), "numbers")

Hello, workshop, let's build something great today!
The embedding model returned 768 numbers


## Step 3. Load the two PDF files from Google Drive

1. In **My Drive**, create a folder called **`airport_rag`**.
2. Put the two PDF files inside it:
   - `GIA-POL-01_Passenger_Service_Standards_Policy_v3.0.pdf`
   - `GIA-SOP-06_Peak_Period_Operations_and_Queue_Escalation_SOP_v1.3.pdf`
3. Run the next cell and allow access to your Drive.

`SimpleDirectoryReader` reads every PDF in the folder. It makes **one document per page**.

In [ ]:
import os
from google.colab import files

PDF_FOLDER = "/content/airport_rag"
os.makedirs(PDF_FOLDER, exist_ok=True)

print("Please select and upload your airport PDF files (e.g., policy and SOP PDFs):")
uploaded = files.upload()

for name, data in uploaded.items():
    with open(os.path.join(PDF_FOLDER, name), "wb") as f:
        f.write(data)

print(f"\nFiles successfully uploaded and saved to: {PDF_FOLDER}")

Please select and upload your airport PDF files (e.g., policy and SOP PDFs):


Saving GIA-POL-01_Passenger_Service_Standards_Policy_v3.0.pdf to GIA-POL-01_Passenger_Service_Standards_Policy_v3.0.pdf
Saving GIA-SOP-06_Peak_Period_Operations_and_Queue_Escalation_SOP_v1.3.pdf to GIA-SOP-06_Peak_Period_Operations_and_Queue_Escalation_SOP_v1.3.pdf

Files successfully uploaded and saved to: /content/airport_rag


In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(input_dir=PDF_FOLDER, required_exts=[".pdf"]).load_data()

print(len(documents), "pages loaded\n")
for d in documents:
    print(d.metadata["file_name"], "| page", d.metadata["page_label"], "|", len(d.text), "characters")

print("\n--- first 500 characters of page 1 ---")
print(documents[0].text[:500])

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/arrow/array.py:50: RuntimeWarning: coroutine 'prepare_chat_params' was never awaited
  from pandas.core import (


3 pages loaded

GIA-POL-01_Passenger_Service_Standards_Policy_v3.0.pdf | page 1 | 2511 characters
GIA-POL-01_Passenger_Service_Standards_Policy_v3.0.pdf | page 2 | 1419 characters
GIA-SOP-06_Peak_Period_Operations_and_Queue_Escalation_SOP_v1.3.pdf | page 1 | 2265 characters

--- first 500 characters of page 1 ---
General International Airport (fictional) | GIA-POL-01 v3.0 - APPROVED
Page 1
PASSENGER SERVICE STANDARDS POLICY
General International Airport (fictional)
Document no.
GIA-POL-01
Version
3.0
Status
APPROVED
Classification
Internal
Effective
1 January 2026
Next review
1 January 2027
Owner
Director, Passenger Experience
Approved by
Executive Committee (decision
EC-2025-041, 9 December 2025)
Supersedes
GIA-SUP-12 (Passenger Service Standards PSS-01 v2.0)
This is a fictional document created for tra


**No Google Drive?** Run this cell instead of the two above. It lets you upload the PDFs straight from your computer.

## Step 4. What is an embedding?

A computer cannot compare the *meaning* of two sentences, but it can compare two lists of numbers.
An **embedding model** reads a piece of text and returns a list of numbers, called a **vector**.
Text with a similar meaning gets a similar vector, even if the words are different.

We measure how close two vectors are with **cosine similarity**: close to **1** means very similar, close to **0** means unrelated.

Look at three sentences. Two say the same thing in different words. One is about something else.

In [ ]:
import numpy as np

sentences = [
    "Passengers wait too long at passport control on Friday evenings.",
    "The immigration queue is very long on Friday nights.",
    "The duty free shop sells perfume and chocolate.",
]
vectors = Settings.embed_model.get_text_embedding_batch(sentences)

print("Numbers per vector:", len(vectors[0]))
print("First 8 numbers of sentence 1:", np.round(vectors[0][:8], 4))

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print("\nSentence 1 vs 2 (same meaning, different words):", round(cosine(vectors[0], vectors[1]), 3))
print("Sentence 1 vs 3 (different topic)              :", round(cosine(vectors[0], vectors[2]), 3))

Numbers per vector: 768
First 8 numbers of sentence 1: [-0.0229 -0.0015  0.0052 -0.0712 -0.0062  0.0062 -0.0084  0.0171]

Sentence 1 vs 2 (same meaning, different words): 0.901
Sentence 1 vs 3 (different topic)              : 0.74


**What to notice.** Sentences 1 and 2 share almost no words, yet they score higher together than with sentence 3.
That is the whole idea of RAG search: we embed the **question**, embed every **chunk** of the documents, and pick the chunks whose vectors are closest to the question.

## Step 5. Build the vector index and save it

`VectorStoreIndex.from_documents` does four things for us:
1. **Split** each page into smaller pieces called **chunks** (nodes)
2. **Embed** every chunk with the embedding model
3. **Store** the chunk text and its vector in a **vector store**
4. We then **save** everything to Drive so we do not pay to embed again

For now we cut the text into **small chunks of 200 tokens** (with 20 tokens shared between neighbours) so you can see many chunks. In Step 7 we test other sizes.
*(LlamaIndex's own default is 1024 tokens, which would turn our three short pages into just three chunks.)*

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter

index = VectorStoreIndex.from_documents(
    documents,
    transformations=[SentenceSplitter(chunk_size=200, chunk_overlap=20)],
    show_progress=True,
)

PERSIST_DIR = "/content/drive/MyDrive/airport_rag_index"      # saved next to your PDF folder
index.storage_context.persist(persist_dir=PERSIST_DIR)
print("Saved to", PERSIST_DIR)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/11 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/airport_rag_index


### Open the saved files

LlamaIndex saved a few small JSON files. Let us see what is in them.

In [ ]:
import os, json

for name in sorted(os.listdir(PERSIST_DIR)):
    size_kb = os.path.getsize(os.path.join(PERSIST_DIR, name)) / 1024
    print(f"{name:<32} {size_kb:8.1f} KB")

default__vector_store.json          120.1 KB
docstore.json                        31.5 KB
graph_store.json                      0.0 KB
image__vector_store.json              0.1 KB
index_store.json                      1.1 KB


In [ ]:
import pandas as pd

# 1) The vector store: one list of numbers for every chunk
vector_store = json.load(open(f"{PERSIST_DIR}/default__vector_store.json"))
embeddings = vector_store["embedding_dict"]
print("Chunks stored:", len(embeddings))

# 2) The docstore: the text of every chunk
docstore = json.load(open(f"{PERSIST_DIR}/docstore.json"))["docstore/data"]

rows = []
for node_id, vector in list(embeddings.items())[:5]:
    node = docstore[node_id]["__data__"]
    rows.append({
        "chunk id": node_id[:8],
        "file": node["metadata"]["file_name"][:18],
        "text (start)": node["text"][:60].replace("\n", " "),
        "numbers": len(vector),
        "first 4 numbers": [round(x, 3) for x in vector[:4]],
    })
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)

Chunks stored: 11


,chunk id,file,text (start),numbers,first 4 numbers
0,f6e03a87,GIA-POL-01_Passeng,General International Airport (fictional) | GIA-POL-01 v3.0,768,"[-0.015, 0.018, 0.008, -0.051]"
1,861748d5,GIA-POL-01_Passeng,Section 1. Purpose and scope 1.1 This policy sets the minimu,768,"[-0.01, 0.0, 0.006, -0.037]"
2,278628f2,GIA-POL-01_Passeng,2.2 Departure security screening: the average wait must not,768,"[-0.004, 0.002, -0.014, -0.054]"
3,89c143fb,GIA-POL-01_Passeng,Minimums for other checkpoints are set in the Counter Staffi,768,"[-0.004, 0.009, 0.015, -0.043]"
4,4a4b0056,GIA-POL-01_Passeng,General International Airport (fictional) | GIA-POL-01 v3.0,768,"[0.003, -0.007, 0.015, -0.059]"


**Read the table.** Each chunk has its **text** in one file and its **vector** in another, linked by the chunk id.
The vector store is nothing magical: a list of numbers per chunk, plus a fast way to find the closest ones.

Because we saved it, we can load it back later **without calling the embedding model again**:

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
index = load_index_from_storage(storage_context)
print("Index loaded from disk. Ready to ask questions.")

Index loaded from disk. Ready to ask questions.


## Step 6. Ask questions

When you ask a question, RAG does this:
1. The question is embedded
2. The **top_k** closest chunks are retrieved
3. The chunks and the question are sent to Gemini, which writes the answer

`similarity_top_k=3` means: bring back the 3 closest chunks. The `ask` function below prints the answer **and** the chunks it used, with their similarity score.

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=3)

def ask(question, engine=query_engine):
    response = engine.query(question)
    print("QUESTION:", question)
    print("ANSWER  :", str(response), "\n")
    print("Chunks used:")
    for n in response.source_nodes:
        preview = n.text.strip().replace("\n", " ")[:90]
        print(f"  score {n.score:.3f} | {n.metadata['file_name'][:14]} p{n.metadata['page_label']} | {preview}")
    print("-" * 100)

ask("What is the maximum average wait for departure immigration?")
ask("How many departure immigration counters must be open in Terminal 1 at peak hours?")
ask("What must the duty manager do when the average wait goes above 20 minutes?")

QUESTION: What is the maximum average wait for departure immigration?
ANSWER  : The maximum average wait for departure immigration must not exceed 15 minutes. 

Chunks used:
  score 0.746 | GIA-POL-01_Pas p1 | 2.2 Departure security screening: the average wait must not exceed 10 minutes. 2.3 Departu
  score 0.715 | GIA-POL-01_Pas p1 | Minimums for other checkpoints are set in the Counter Staffing and Roster Standard. 3.3 If
  score 0.701 | GIA-SOP-06_Pea p1 | 2.3 Level 3: individual waits above 45 minutes. The duty manager activates priority lanes 
----------------------------------------------------------------------------------------------------
QUESTION: How many departure immigration counters must be open in Terminal 1 at peak hours?
ANSWER  : During peak hours, a minimum of 9 departure immigration counters must be open in Terminal 1. 

Chunks used:
  score 0.767 | GIA-POL-01_Pas p1 | 2.2 Departure security screening: the average wait must not exceed 10 minutes. 2.3 Departu
  score

Now two harder ones. The first needs **two documents**. The second asks about something that is **not in the documents** at all. Watch what the model does.

In [ ]:
ask("On a Friday evening only 5 counters are open in Terminal 1. Which rule is not met, and how many reserve counters exist?")
ask("What is the daily parking tariff at the airport?")

QUESTION: On a Friday evening only 5 counters are open in Terminal 1. Which rule is not met, and how many reserve counters exist?
ANSWER  : During peak hours (which occur every day of the week from 17:00 to 22:59, covering Friday evenings), Terminal 1 is required to have a minimum of 9 departure immigration counters open. Since only 5 counters are open, this rule is not met. 

Regarding reserve counters, Terminal 1 has a total of 14 departure immigration counters installed. With 5 counters open, there are 9 unrostered counters available as reserve capacity. 

Chunks used:
  score 0.691 | GIA-SOP-06_Pea p1 | 2.3 Level 3: individual waits above 45 minutes. The duty manager activates priority lanes 
  score 0.678 | GIA-POL-01_Pas p1 | 2.2 Departure security screening: the average wait must not exceed 10 minutes. 2.3 Departu
  score 0.674 | GIA-POL-01_Pas p1 | Section 1. Purpose and scope 1.1 This policy sets the minimum service levels that every pa
----------------------------------------

**Discuss.** For the parking question, does the model say it does not know, or does it invent something? We fix this with a rule prompt in Step 8.

## Step 7. Chunking and top_k: test, do not guess

**Chunking** is how we cut the documents before embedding. Two knobs matter:
- **chunk size**: how many tokens in each chunk (for example 200, 300, 600)
- **chunk overlap**: how many tokens two neighbouring chunks share, so a sentence cut at the edge still appears whole in one of them

Three kinds of chunking:

| Type | How it cuts | Good for | Risk |
|---|---|---|---|
| **Token (fixed size)** | Every N tokens, no matter what | Speed, very simple | Cuts sentences and rules in half |
| **Sentence** | Around N tokens, but only at sentence ends | Most documents (this is the LlamaIndex default) | Needs a sensible size |
| **Semantic** | Where the *meaning* changes, using embeddings | Long, mixed-topic documents | Slower, uses more embedding calls |

**top_k** is how many chunks we hand to the model. Too small and we miss part of the answer. Too large and we fill the prompt with noise.

*A note on "top p" and "top k" you may know from model settings.* Those are **sampling** settings for the model's writing (together with temperature). On Gemini 3.5 Flash-Lite Google has deprecated them and the API ignores them. The only top-k that matters here is the **retrieval** `similarity_top_k` above.

### The scoreboard
We write 6 questions where we know which text holds the answer. A search **hits** if the right text is inside the top 1, 3 or 5 chunks.

In [ ]:
# (question, a short phrase that must appear in the right chunk)
GOLDEN = [
    ("What is the maximum average wait for departure immigration?", "exceed 15 minutes"),
    ("How many departure immigration counters must be open in Terminal 1 at peak hours?", "9 counters in terminal 1"),
    ("What happens when the average wait is above 20 minutes?", "above 20 minutes"),
    ("How many departure immigration counters are installed in Terminal 1?", "14 departure immigration counters"),
    ("When do smart gates start at Terminal 1 departures?", "1 july 2026"),
    ("How many percent of passengers may wait more than 30 minutes in a peak hour?", "5 percent"),
]

def clean(text):
    return " ".join(text.lower().split())      # lower case, one space between words

def score_index(test_index):
    """Return the share of questions answered correctly within the top 1, 3 and 5 chunks."""
    retriever = test_index.as_retriever(similarity_top_k=5)
    hits = {1: 0, 3: 0, 5: 0}
    for question, phrase in GOLDEN:
        found_at = None
        for rank, n in enumerate(retriever.retrieve(question), start=1):
            if phrase in clean(n.text):
                found_at = rank
                break
        for k in hits:
            if found_at is not None and found_at <= k:
                hits[k] += 1
    return {f"top_{k}": f"{hits[k]}/{len(GOLDEN)}" for k in hits}

In [ ]:
from llama_index.core.node_parser import TokenTextSplitter, SentenceSplitter

CONFIGS = {
    "Token 200":    TokenTextSplitter(chunk_size=200, chunk_overlap=20),
    "Sentence 200": SentenceSplitter(chunk_size=200, chunk_overlap=20),
    "Sentence 300": SentenceSplitter(chunk_size=300, chunk_overlap=30),
    "Sentence 600": SentenceSplitter(chunk_size=600, chunk_overlap=60),
}

results, test_nodes = [], {}
for name, splitter in CONFIGS.items():
    nodes = splitter.get_nodes_from_documents(documents)
    test_index = VectorStoreIndex(nodes)
    test_nodes[name] = nodes
    results.append({"chunking": name, "chunks": len(nodes), **score_index(test_index)})

pd.DataFrame(results)

,chunking,chunks,top_1,top_3,top_5
0,Token 200,12,3/6,5/6,5/6
1,Sentence 200,11,4/6,6/6,6/6
2,Sentence 300,8,4/6,5/6,5/6
3,Sentence 600,4,0/6,6/6,6/6


**Read the scoreboard.** `top_1` is the strictest: the right chunk came first. `top_5` is the most generous.
Fewer chunks means bigger chunks; more chunks means smaller, more focused chunks.

Now look at **how the cuts differ**. Same size, two kinds of chunking:

In [ ]:
for name in ["Token 200", "Sentence 200"]:
    print("=" * 30, name, "=" * 30)
    for node in test_nodes[name][1:3]:
        print(node.text.strip())
        print("." * 60)

============================== Token 200 ==============================
a fictional document created for training purposes.
Section 1. Purpose and scope
1.1 This policy sets the minimum service levels that every passenger should experience at the airport,
from arrival at the terminal to boarding or leaving.
1.2 It applies to Terminal 1 and Terminal 2 and to the airport operator, border agencies working in the
terminals, security contractors and ground handlers.
1.3 Related documents: Counter Staffing and Roster Standard (GIA-STD-03), Peak-Period Operations
and Queue Escalation SOP (GIA-SOP-06), Complaint Handling Procedure (GIA-SOP-05), KPI and Data
Definitions Manual (GIA-MAN-08).
Section 2. Processing time targets
2.1 Departure check-in: the average
............................................................
2. Processing time targets
2.1 Departure check-in: the average wait must not exceed 10 minutes.
2.2 Departure security screening: the average wait must not exceed 10 minutes.
2.

**Notice.** The token splitter can stop in the middle of a sentence. The sentence splitter stops at a sentence end.

### Optional: semantic chunking
Semantic chunking asks the embedding model where the topic changes. It makes more embedding calls, so it is slower.

In [ ]:
from llama_index.core.node_parser import SemanticSplitterNodeParser

semantic = SemanticSplitterNodeParser(
    buffer_size=1,
    breakpoint_percentile_threshold=95,
    embed_model=Settings.embed_model,
)
semantic_nodes = semantic.get_nodes_from_documents(documents)
semantic_index = VectorStoreIndex(semantic_nodes)

print("Semantic chunks:", len(semantic_nodes))
print(score_index(semantic_index))

Semantic chunks: 8
{'top_1': '3/6', 'top_3': '5/6', 'top_5': '6/6'}


### Now top_k: how many chunks should the model see?

This question needs rules from **two documents** (the policy and the procedure). Try `top_k` 1, 3 and 6 and compare the answers.

In [ ]:
question = "What must the duty manager do when the average departure immigration wait is above 30 minutes for 3 consecutive hours?"

for k in [1, 3, 6]:
    engine = index.as_query_engine(similarity_top_k=k)
    response = engine.query(question)
    files = sorted({n.metadata["file_name"][:14] for n in response.source_nodes})
    print(f"top_k = {k}  |  documents used: {files}")
    print(str(response))
    print("-" * 100)

top_k = 1  |  documents used: ['GIA-SOP-06_Pea']
When the average departure immigration wait is above 30 minutes for 3 consecutive hours, the duty manager must:

1. Request additional border staff immediately.
2. Record the incident.
3. Notify the Director, Airport Operations.
----------------------------------------------------------------------------------------------------
top_k = 3  |  documents used: ['GIA-POL-01_Pas', 'GIA-SOP-06_Pea']
When the average departure immigration wait exceeds 30 minutes for 3 consecutive hours, the duty manager must request additional border staff immediately, record the incident, and notify the Director, Airport Operations.
----------------------------------------------------------------------------------------------------
top_k = 6  |  documents used: ['GIA-POL-01_Pas', 'GIA-SOP-06_Pea']
When the average departure immigration wait exceeds 30 minutes for 3 consecutive hours, the airport duty manager must immediately request additional border staff and

**Discuss.** With `top_k = 1` the model only sees one chunk. With a larger number it can combine the policy and the procedure.
A good starting point is **3 to 5**. Always measure with your own questions, as we did above.

## Step 8. Lock in the settings and add a rule prompt

Now choose the settings that worked best on your scoreboard. We start with **sentence chunking, size 300, overlap 30, top_k 3**. Change them if your table says otherwise.

We also add a short **prompt** that tells the model how to behave: use only the context, cite the document and section, and admit when the answer is not there. This is the same idea as the system prompt from Day 1.

In [ ]:
CHUNK_SIZE = 300
CHUNK_OVERLAP = 30
TOP_K = 3

final_index = VectorStoreIndex.from_documents(
    documents,
    transformations=[SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)],
)
final_index.storage_context.persist(persist_dir=PERSIST_DIR + "_final")
print("Final index built with", len(final_index.docstore.docs), "chunks")

Final index built with 8 chunks


In [ ]:
from llama_index.core import PromptTemplate

qa_prompt = PromptTemplate(
    "You are the Airport Operations Assistant.\n"
    "Answer using ONLY the context below.\n"
    "Cite the document number and section for every rule, for example [GIA-POL-01, 2.3].\n"
    "If the context does not contain the answer, say exactly: I could not find this in the documents.\n"
    "Keep the answer under 120 words.\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer: "
)

final_engine = final_index.as_query_engine(similarity_top_k=TOP_K, text_qa_template=qa_prompt)

ask("On a Friday evening only 5 counters are open in Terminal 1. Which rule is not met?", engine=final_engine)
ask("What is the daily parking tariff at the airport?", engine=final_engine)

QUESTION: On a Friday evening only 5 counters are open in Terminal 1. Which rule is not met?
ANSWER  : Based on the provided documents, during peak hours (17:00 to 22:59 every day of the week, in both terminals), a minimum of 9 departure immigration counters must be open in Terminal 1 [GIA-POL-01, 3.1, 3.2]. Having only 5 counters open does not meet this requirement [GIA-POL-01, 3.2]. 

Chunks used:
  score 0.708 | GIA-POL-01_Pas p1 | 1.3 Related documents: Counter Staffing and Roster Standard (GIA-STD-03), Peak-Period Oper
  score 0.691 | GIA-SOP-06_Pea p1 | 4.2 Smart gates reduce load on staffed counters but do not replace the minimum counters in
  score 0.683 | GIA-SOP-06_Pea p1 | 2.2 Level 2: the average departure immigration wait is above 30 minutes for 3 consecutive 
----------------------------------------------------------------------------------------------------
QUESTION: What is the daily parking tariff at the airport?
ANSWER  : I could not find this in the documents. 

Chun

**Compare** the parking answer with Step 6. The prompt turns a confident guess into an honest "I could not find this in the documents".

## Step 9. Put a Gradio chat window on top

A notebook is for builders. Managers want a window where they type a question. Gradio gives us one in a few lines.
The app shows the **answer** and the **sources** it used, so people can check the facts.

In [ ]:
import gradio as gr

def answer_question(question):
    if not question or not question.strip():
        return "Please type a question.", ""
    response = final_engine.query(question)
    sources = "\n\n".join(
        f"[{i}] {n.metadata['file_name']} (page {n.metadata['page_label']}, score {n.score:.2f})\n{n.text.strip()}"
        for i, n in enumerate(response.source_nodes, start=1)
    )
    return str(response), sources

with gr.Blocks(title="Airport Policy Assistant") as demo:
    gr.Markdown("## Airport Policy Assistant\nAsk about passenger service standards and queue escalation.")
    question_box = gr.Textbox(label="Your question", placeholder="For example: How many counters must be open in Terminal 1 at peak hours?")
    ask_button = gr.Button("Ask")
    answer_box = gr.Textbox(label="Answer", lines=6)
    sources_box = gr.Textbox(label="Sources used", lines=12)
    gr.Examples(
        examples=[
            "What is the maximum average wait for departure immigration?",
            "What must the duty manager do above 20 minutes?",
            "When do smart gates start at Terminal 1?",
            "What is the daily parking tariff?",
        ],
        inputs=question_box,
    )
    ask_button.click(answer_question, question_box, [answer_box, sources_box])
    question_box.submit(answer_question, question_box, [answer_box, sources_box])

demo.launch(share=True)

/usr/local/lib/python3.13/dist-packages/uvicorn/server.py:86: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
Exception in thread Thread-6 (run):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/server.py", line 86, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
TypeError: _patch_asyncio.<locals>.run() got an unexpected keyword argument 'loop_factory'
Exception in thread Thread-7 (run):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/u

KeyboardInterrupt: 

## Your turn

1. **Change the settings.** Set `CHUNK_SIZE = 200` or `600` and `TOP_K = 5`, rebuild the final index, and ask the Friday question again. What changes?
2. **Break it.** Ask something that is not in the documents. Ask in other words ("passport control" instead of "immigration"). Does retrieval still find the right chunk?
3. **Add a document.** Put a third PDF in the Drive folder, rebuild from Step 3, and ask a question that needs it.
4. **Write one more golden question** with its answer phrase and see how the scoreboard changes.

## What we carry forward

- **Embedding** = text turned into a vector. Close vectors mean close meaning.
- A **vector store** is chunk text plus chunk vectors. We **save** it so we embed only once.
- **Chunk size and overlap** change what the search can find. Test them on questions you know the answer to.
- **top_k** is the number of chunks the model sees. Start at 3 to 5 and measure.
- A short **rule prompt** makes the model cite its sources and admit when it does not know.